# 03 - Python Batch Loader (تحميل الملفات الصغيرة)

هذا النوتبوك يغطي القسم **6.3** من وثيقة التكليف: تحميل الملفات الصغيرة عبر Python بأسلوب Streaming.

**متطلبات محققة في `src/batch_loader.py`:**
- قراءة CSV سطرًا سطرًا عبر `csv.DictReader` (بدون `list(reader)` وبدون تحميل الملف كاملاً بالذاكرة)
- تجميع السجلات في دفعات (`BATCH_SIZE` من الإعدادات) ثم `insert_many`
- طباعة رقم الدفعة، عدد السجلات، الزمن، ومعدل الإدخال لكل دفعة
- معالجة خطأ الدفعة وتسجيله دون إخفاء السبب (`BulkWriteError` مُعالَجة بوضوح)

**طبقة ELT (القسم 6.5):** كل سجل يُحمَّل كما وصل (`record_raw`) إلى `orders_raw`، مع بيانات المصدر والتشغيل: `id_run, file_source, number_row_source, at_ingested, engine_used`. **لا يوجد أي تنظيف أو فلترة هنا** — هذا يأتي لاحقًا في نوتبوك 05.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
sys.path.append(str(PROJECT_ROOT / "src"))

from config import settings
from file_router import choose_engine, print_decision
from batch_loader import load_batch
from pymongo import MongoClient

print(f"BATCH_SIZE الحالي: {settings.BATCH_SIZE}")

BATCH_SIZE الحالي: 5000


## الخطوة 1: نستدعي الـ Router أولاً (نفس نقطة الدخول الموحدة)

حسب معمارية المشروع، لا نستدعي `batch_loader` مباشرة بمعزل عن الـ Router — الـ Router هو من يقرر المحرك ويولّد `id_run` لعملية التشغيل.

In [2]:
sample_exists = settings.SAMPLE_FILE_PATH.exists()

if sample_exists:
    decision = choose_engine(settings.SAMPLE_FILE_PATH)
    print_decision(decision)
    assert decision["engine"] == "python_batch"
else:
    print("العينة غير موجودة. ارجع لنوتبوك 01 وأنشئها أولاً.")

== قرار الـ Router ==
id_run       : run_20260822T185741_428ea020
file_name    : orders_small_sample.csv
file_size_mb : 41.765
threshold_mb : 200.0
engine       : python_batch
reason       : حجم الملف (41.77 MB) أقل من أو يساوي الحد الفاصل (200.0 MB) => Batch Python كافٍ وأسرع بدون تكلفة تشغيل Spark.


## الخطوة 2: تشغيل Batch Loader فعليًا على العينة

سيُحمَّل كل سجل كما هو (بدون تنظيف) إلى `orders_raw` في MongoDB.

In [3]:
if sample_exists:
    summary = load_batch(
        input_path=settings.SAMPLE_FILE_PATH,
        id_run=decision["id_run"],
        batch_size=settings.BATCH_SIZE,
    )

    print("\n== ملخص التشغيل ==")
    for k, v in summary.items():
        if k != "batch_errors":
            print(f"{k}: {v}")
    if summary["batch_errors"]:
        print(f"\n[تنبيه] عدد الدفعات التي واجهت أخطاء: {len(summary['batch_errors'])}")
        for err in summary["batch_errors"]:
            print(err)

دفعة    1 | سجلات:   5000/5000   | زمن الدفعة: 0.098s | معدل: 50886.06 سجل/ثانية | الزمن الكلي: 0.15s
دفعة    2 | سجلات:   5000/5000   | زمن الدفعة: 0.045s | معدل: 110287.61 سجل/ثانية | الزمن الكلي: 0.27s
دفعة    3 | سجلات:   5000/5000   | زمن الدفعة: 0.075s | معدل: 67098.34 سجل/ثانية | الزمن الكلي: 0.39s
دفعة    4 | سجلات:   5000/5000   | زمن الدفعة: 0.046s | معدل: 109867.56 سجل/ثانية | الزمن الكلي: 0.48s
دفعة    5 | سجلات:   5000/5000   | زمن الدفعة: 0.052s | معدل: 95592.75 سجل/ثانية | الزمن الكلي: 0.57s
دفعة    6 | سجلات:   5000/5000   | زمن الدفعة: 0.065s | معدل: 76592.06 سجل/ثانية | الزمن الكلي: 0.67s
دفعة    7 | سجلات:   5000/5000   | زمن الدفعة: 0.056s | معدل: 88990.2 سجل/ثانية | الزمن الكلي: 0.79s
دفعة    8 | سجلات:   5000/5000   | زمن الدفعة: 0.062s | معدل: 81223.89 سجل/ثانية | الزمن الكلي: 0.90s
دفعة    9 | سجلات:   5000/5000   | زمن الدفعة: 0.046s | معدل: 109100.52 سجل/ثانية | الزمن الكلي: 0.99s
دفعة   10 | سجلات:   5000/5000   | زمن الدفعة: 0.048s | معدل: 103437.91 سجل/ثاني

## الخطوة 3: التحقق من MongoDB مباشرة

نتأكد أن السجلات فعليًا وصلت إلى `orders_raw`، وأن العدد يطابق `read_rows` (شرط الاتساق الأساسي في القسم 6.11: لا فقدان لأي سجل).

In [4]:
if sample_exists:
    client = MongoClient(settings.MONGO_URI)
    collection = client[settings.MONGO_DB_NAME][settings.COLLECTION_RAW]

    count_for_this_run = collection.count_documents({"id_run": decision["id_run"]})
    print(f"عدد السجلات في orders_raw لهذا التشغيل (id_run={decision['id_run']}): {count_for_this_run}")
    print(f"عدد السجلات المقروءة من الملف (read_rows): {summary['read_rows']}")

    assert count_for_this_run == summary["read_rows"], "⚠️ عدم تطابق! هناك سجلات مفقودة."
    print("\n[تحقق ناجح] كل السجلات المقروءة وصلت فعليًا إلى orders_raw، بدون أي فقدان.")

    print("\n== نموذج أول وثيقة في orders_raw ==")
    sample_doc = collection.find_one({"id_run": decision["id_run"]})
    for k, v in sample_doc.items():
        print(f"{k}: {v}")

    client.close()

عدد السجلات في orders_raw لهذا التشغيل (id_run=run_20260822T185741_428ea020): 100000
عدد السجلات المقروءة من الملف (read_rows): 100000

[تحقق ناجح] كل السجلات المقروءة وصلت فعليًا إلى orders_raw، بدون أي فقدان.

== نموذج أول وثيقة في orders_raw ==
_id: 6a89f18f278028cbc363f875
id_run: run_20260822T185741_428ea020
file_source: D:\midterm-data-pipeline\data\orders_small_sample.csv
number_row_source: 1
at_ingested: 2026-08-22T18:59:27.379752+00:00
engine_used: python_batch
record_raw: {'\ufefforder_id': 'طلب-100000', 'order_date': '2025-02-24T21:29:00', 'status': 'مؤكد', 'customer_id': 'عميل-0', 'customer_name': 'محمد علي', 'customer_phone': '702390941', 'customer_email': 'user141764@example.com', 'city': 'تعز', 'district': 'شعوب', 'delivery_type': 'سريع', 'delivery_cost': '5000.0', 'payment_method': 'محفظة إلكترونية', 'payment_status': 'تم الدفع', 'payment_amount': '769000.0', 'currency': 'YER', 'total_amount': '769000.0', 'items_json': '[{"sku":"SKU-1010","name":"هاتف سامسونج A54","qty"

## ملاحظة مهمة: نمط ELT

لاحظ في الوثيقة أعلاه أن `record_raw` يحتوي على البيانات **كما وصلت تمامًا** — بما في ذلك أي قيم غير نظيفة (أرقام عربية، عملات نصية، تواريخ غير موحدة، إلخ). **لم يتم أي تنظيف هنا عن قصد**، لأن هذا هو جوهر نمط ELT المطلوب في القسم 6.5:

> يجب تحميل البيانات كاملة أولًا (Raw) قبل تطبيق قواعد التنظيف.

## الخطوة القادمة

**`04_pyspark_loader.ipynb`** — بناء محمّل الملفات الكبيرة باستخدام PySpark، وتشغيله فعليًا على الملف الضخم (~12.6 GB) لإثبات المسار الثاني من الـ Router.